# Comparison of satellite images and learned embeddings for land cover mapping in the Brazilian Amazon

## Models
1) Deep-learning approach: resnet50 trained on Sentinel-2 satellite images
2) Deep-learning approach: fully connected convnet trained on AE embeddings
3) Machine-learning approach: Random Forest classifier trained on a subset of pixels from AE embeddings

## Inference 
This notebook allows to run inference
1) Loads trained parameters from our models
2) Loads data from the test set
3) Applies the models on 2 images from the test set
4) Displays them

## Load Models and parameters
should take about 1m 18s

In [1]:
from utils import load_model_sentinel
from utils import load_model_AE
from utils import load_model_AE_RF

# Load models
model_s2=load_model_sentinel("final_models/s2_final.pth")
model_AE=load_model_AE("final_models/AE_final.pth")
model_AE_RF, scaler=load_model_AE_RF()
print("Models successfully loaded")

Models successfully loaded


## Load 2 selected samples from the Test Set
should take <2s

In [4]:
import numpy as np
import rasterio

s2_path = "Test_for_inference/S2/"
ae_path = "Test_for_inference/AE/"
gt_path = "Test_for_inference/groundtruth/"

#fnames = ["test_010040.tif", "test_010074.tif", "test_010087.tif", "test_010116.tif", "test_010143.tif", "test_010219.tif", "test_010351.tif", "test_010489.tif"]
#since prediction of the S2 model takes a while on CPU, limit to 2 files

fnames = ["test_010657.tif", "test_010074.tif"]
LABEL_REMAP = {3:1, 4:2, 6:3, 9:4, 11:5, 12:6, 15:7, 18:8, 24:9, 25:10, 30:11, 33:12}
data = []
for fname in fnames:
    s2_img_path = s2_path + fname
    ae_img_path = ae_path + fname
    lbl_path = gt_path + fname
    # Read multispectral S2 image
    with rasterio.open(s2_img_path) as src:
        s2_img = src.read().astype(np.float32)  # (C, H, W)
    # Read AE embeddings
    with rasterio.open(ae_img_path) as src:
        ae_img = src.read().astype(np.float32)  # (C, H, W)
    # Read label mask (single channel)
    with rasterio.open(lbl_path) as src:
        label_raw = src.read(1).astype(np.int32)

    # Remap labels to contiguous indices
    label = np.zeros_like(label_raw, dtype=np.int32)
    for old_id, new_id in LABEL_REMAP.items():
        label[label_raw == old_id] = new_id
    data.append([s2_img, ae_img, label])

for fname in fnames:
    print(f"Image {fname} successfully loaded")

Image test_010657.tif successfully loaded
Image test_010074.tif successfully loaded


# Inference: run forward pass through each model

In [5]:
# First define the inference function:
import torch
@torch.no_grad() #without gradients
def inference(dataset, idx, model, modality="s2", scaler=None):
    import tqdm
    import numpy as np
    import torchvision.transforms as T
    # Only runs on CPU because of the limited amount of images to predict
    model.eval()
    predictions = []
    for id in tqdm(idx, desc=f"Forward pass of {modality} model"):
        #load image and ground truth
        print(idx)
        print(id)
        img, label = dataset[id]
        #prepare input and noramlize
        x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)
        #normalization
        s2=np.load("mean_std/s2.npy") # Mean and std of Sentinel-2
        s2_mean, s2_std = s2[0], s2[1]
        s2_mean, s2_std = torch.tensor(s2_mean), torch.tensor(s2_std)
        s2_normalize = T.Normalize(s2_mean, s2_std)
        AE=np.load("mean_std/AE.npy") # Mean and std of AE-Embeddings
        AE_mean, AE_std = AE[0], AE[1]
        AE_mean, AE_std = torch.tensor(AE_mean), torch.tensor(AE_std)
        AE_normalize = T.Normalize(AE_mean, AE_std) 
        #inference
        if modality == "s2":
            x = s2_normalize(x)
            #forward pass
            outputs = model(x)
            pred = outputs["out"].argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE":
            x = AE_normalize(x)
            #forward pass
            pred = model(x)
            pred = pred.argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE_RF":
            x = img.astype(np.float32)
            x = scaler.transform(x)
            #prediction
            pred = model.predict(x)
        else:
            print("Modality must be 'AE', 'AE_RF' or 's2'")
        predictions.append(pred)
    return predictions

sould take about 3 minutes to run on CPU (due to the size of the Sentinel-2 resnet model)

In [7]:
#from utils import inference
import numpy as np
import torch
@torch.no_grad() #without gradients
def inference(data, model, modality="s2", num_image=2, scaler=None):
    import numpy as np
    import torchvision.transforms as T
    # Only runs on CPU because of the limited amount of images to predict
    
    #normalization
    s2=np.load("mean_std/s2.npy") # Mean and std of Sentinel-2
    s2_mean, s2_std = s2[0], s2[1]
    s2_mean, s2_std = torch.tensor(s2_mean), torch.tensor(s2_std)
    s2_normalize = T.Normalize(s2_mean, s2_std)
    AE=np.load("mean_std/AE.npy") # Mean and std of AE-Embeddings
    AE_mean, AE_std = AE[0], AE[1]
    AE_mean, AE_std = torch.tensor(AE_mean), torch.tensor(AE_std)
    AE_normalize = T.Normalize(AE_mean, AE_std) 
    predictions = []
    for id in range(num_image):
        #load image and ground truth
        s2_img, ae_img, label = data[id]
        #inference
        if modality == "s2":
            print(f"Sentinel-2 Model: computing predictions ({id+1}/{num_image})")
            model.eval()
            x = torch.from_numpy(s2_img).unsqueeze(0)
            x = s2_normalize(x)
            #forward pass
            outputs = model(x)
            pred = outputs["out"].argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE":
            print(f"AE Deep-Learning Model: computing predictions ({id+1}/{num_image})")
            model.eval()
            x = torch.from_numpy(ae_img).unsqueeze(0)
            x = AE_normalize(x)
            #forward pass
            pred = model(x)
            pred = pred.argmax(1).squeeze(0).numpy().astype(np.uint8)
        elif modality == "AE_RF":
            print(f"AE Random Forest Model: computing predictions ({id+1}/{num_image})")
            ae_img = np.transpose(ae_img, (1, 2, 0))
            H, W, C = ae_img.shape
            x = ae_img.reshape(-1, C)
            #normalization
            x = scaler.transform(x)
            #prediction
            pred = model.predict(x)
            #reshape back to (H,W)
            pred = pred.reshape(H,W)
        else:
            print("Modality must be 'AE', 'AE_RF' or 's2'")
        predictions.append(pred)
    return predictions

num_image=len(data)
prediction_s2 = inference(data, model_s2,num_image=num_image, modality="s2")
print("--- Sentinel-2 Model : predictions computed ---")
prediction_AE = inference(data, model_AE,num_image=num_image, modality="AE")
print("--- AE Deep-Learning Model : predictions computed ---")
prediction_AE_RF = inference(data, model_AE_RF,num_image=num_image, modality="AE_RF", scaler=scaler)
print("--- AE Random Forest Model : predictions computed ---")

Sentinel-2 Model: computing predictions (0/2)


Sentinel-2 Model: computing predictions (1/2)
Sentinel-2 Model : predictions computed
AE Deep-Learning Model: computing predictions (0/2)
AE Deep-Learning Model: computing predictions (1/2)
AE Deep-Learning Model : predictions computed
AE Random Forest Model: computing predictions (0/2)
(47, 47, 64)
(2209, 64)
(2209,)
(47, 47)
AE Random Forest Model: computing predictions (1/2)
(47, 47, 64)
(2209, 64)
(2209,)
(47, 47)
AE Random Forest Model : predictions computed


# Visualisation

In [9]:

# Define visualisation function (interactive list)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from ipywidgets import interact

REMAPPED_ID_TO_NAME = {1: "Forest", 2: "Savanna", 3: "Grassland", 4: "Wetland", 5: "Cropland", 6: "Pasture", 7: "Urban",
    8: "Mining", 9: "Water", 10: "Bare Soil", 11: "Shrubland", 12: "Other"}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys())
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)

def visualize_predictions(data, pred_s2, pred_ae, pred_ae_rf):
    
    @interact(idx=range(len(data)))
    def show(idx=0):
        s2_img, ae_img, gt_label = data[idx]

        # Build RGB from Sentinel-2 (bands 4,3,2)
        r = s2_img[3]
        g = s2_img[2]
        b = s2_img[1]
        rgb = np.stack([r, g, b], axis=-1)
        rgb = rgb.astype(np.float32)
        rgb /= np.percentile(rgb, 99)
        rgb = np.clip(rgb, 0, 1)

        # Predictions
        p_s2 = pred_s2[idx]
        p_ae = pred_ae[idx]
        p_ae_rf = pred_ae_rf[idx]

        # Plot
        fig, axes = plt.subplots(1, 5, figsize=(22, 6))

        axes[0].imshow(rgb)
        axes[0].set_title("RGB")
        axes[0].axis("off")

        axes[1].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")

        axes[2].imshow(p_s2, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[2].set_title("Sentinel‑2 Prediction")
        axes[2].axis("off")

        axes[3].imshow(p_ae, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[3].set_title("AE Prediction")
        axes[3].axis("off")

        axes[4].imshow(p_ae_rf, cmap=CMAP, vmin=0, vmax=N_CLASSES)
        axes[4].set_title("AE‑RF Prediction")
        axes[4].axis("off")

        # Legend
        unique_labels = np.unique(
            np.concatenate([
                np.unique(gt_label),
                np.unique(p_s2),
                np.unique(p_ae),
                np.unique(p_ae_rf)
            ])
        )

        legend_patches = []
        for class_id in unique_labels:
            if class_id == 0:
                name = "Background / Ignored"
            else:
                name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")
            legend_patches.append(mpatches.Patch(color=CMAP(class_id), label=name))

        fig.legend(handles=legend_patches, bbox_to_anchor=(1.05, 0.5), loc="center left")
        plt.tight_layout()
        plt.show()


/tmp/ipykernel_806749/1020666025.py:11: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


In [10]:
visualize_predictions(data, prediction_s2, prediction_AE, prediction_AE_RF)


interactive(children=(Dropdown(description='idx', options=(0, 1), value=0), Output()), _dom_classes=('widget-i…